# 05 — Model Evaluation & Sample Inference

Evaluates the saved CNN-LSTM checkpoint on `X_test` / `y_test`, exports confusion matrix, ROC curves, JSON metrics, and classification report, and runs a sample window prediction.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import load_config
from src.data import load_processed_splits
from src.evaluation import evaluate_model
from src.inference import ActivityPredictor

cfg = load_config(PROJECT_ROOT / "configs" / "config.yaml")
splits = load_processed_splits(PROJECT_ROOT / cfg["data"]["processed_dir"])

predictor = ActivityPredictor.from_checkpoint(
    checkpoint_path=PROJECT_ROOT / cfg["training"]["checkpoint_path"],
    device=str(cfg["training"].get("device", "auto")),
)

metrics = evaluate_model(
    model=predictor.model,
    X_test=splits["X_test"],
    y_test=splits["y_test"],
    class_names=cfg["data"]["class_names"],
    metrics_dir=PROJECT_ROOT / cfg["results"]["metrics_dir"],
    plots_dir=PROJECT_ROOT / cfg["results"]["plots_dir"],
    reports_dir=PROJECT_ROOT / cfg["results"]["reports_dir"],
    device=predictor.device,
)
print("Accuracy:", metrics["accuracy"], "Macro F1:", metrics["macro_f1"], "Macro AUC:", metrics["macro_auc"])

sample_pred = predictor.predict_window(splits["X_test"][0], apply_scaler=False)
print("Sample window prediction:", sample_pred["label"], f"({sample_pred['confidence']:.2%})")